# 02 — Preparando os dados

Um LLM só é tão bom quanto o texto que você dá a ele. Antes de qualquer RAG,
os dados precisam estar **limpos e consistentes**:

- valores em formatos misturados (`"123,45"`, `"- 2.345,67"`, `12.5`) viram `float`;
- as 22 faturas viram uma tabela só;
- transações repetidas entre faturas são removidas (mas dois cafés iguais no mesmo dia continuam!);
- cada linha ganha um `tipo`: `gasto` (positivo) ou `credito` (pagamentos, estornos, descontos).

Toda essa lógica mora em `src/faturas/dados.py` e tem testes em `tests/test_dados.py`.

In [ ]:
from pathlib import Path

import pandas as pd

from faturas.dados import carregar_faturas, parse_amount

PASTA = Path("../data")
df = carregar_faturas(PASTA)
df.head()

## Como o `parse_amount` interpreta cada formato

In [ ]:
for exemplo in ["123,45", "- 2.345,67", "1.234,56", "R$\xa067,89", 12.5]:
    print(f"{exemplo!r:>16} -> {parse_amount(exemplo)}")

In [ ]:
df.info()

## Quantas linhas, quantas duplicatas

Comparamos o total bruto de linhas dos CSVs com o resultado limpo.

In [ ]:
bruto = sum(len(pd.read_csv(f)) for f in sorted(PASTA.glob("Nubank_*.csv")))
print(f"linhas nos CSVs: {bruto}")
print(f"linhas após limpeza: {len(df)}")
print(f"duplicatas removidas: {bruto - len(df)}")
print(f"período: {df['date'].min():%d/%m/%Y} a {df['date'].max():%d/%m/%Y}")

## Gastos vs créditos

Repare que somar tudo daria quase zero — os pagamentos da fatura (`credito`) anulam os gastos.
Por isso, perguntas como "quanto gastei?" devem olhar **só** para `tipo == "gasto"`.
Guarde isso: é exatamente o tipo de detalhe que um LLM erra se ninguém contar a ele.

In [ ]:
df.groupby("tipo")["amount"].agg(["count", "sum"]).round(2)

## Prévia da etapa 03: uma transação como texto

Modelos de linguagem leem **texto**, não DataFrames. No RAG, cada transação vira
uma linha de texto (um "documento") que será transformada em embedding.

In [ ]:
linha = df.iloc[-1]
print(f"{linha['date']:%Y-%m-%d} | {linha['title']} | R$ {linha['amount']:.2f} | {linha['tipo']}")